## CUAD RAG — Scale-up baseline (Experiment N config, N=100)

Experiment N (HyDE + nomic + mistral-small3.2:24b + PROMPT_V2) was the best config found on N=20,
n_runs=3: Rel 0.540 / Util 0.366 / Comp 0.581 / Adh 30%. That N=20 result has too much sample
variance to draw firm conclusions from (see Retrieval_Strategy.md — "N=5 is too small" applies at
N=20 too, just less severely). This notebook re-runs the identical config at N=100, n_runs=1 to get
a stable baseline before trying anything new (per Retrieval_Strategy.md: for final/full-dataset runs,
drop n_runs to 1 — the dataset itself becomes the variance-reduction mechanism).

In [1]:
import os
import sys
import subprocess
import time

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu datasets pandas -q
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull nomic-embed-text-v2-moe:latest
    !ollama pull llama3.1:8b-instruct-q4_K_M
    !ollama pull mistral-small3.2:24b

In [2]:
%load_ext autoreload
%autoreload 2

In [18]:
import os
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import PROMPT_V2
from reliablerag.chunking import SentenceTextSplitter
from reliablerag.env import load_secrets
from reliablerag.experiment import evaluate_results, run_rag_experiment
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import get_hyde_retriever

### 1. Configuration

Model names and paths are loaded from `.env`. Fallback defaults are used if not set.

In [4]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
GENERATOR_MODEL    = os.environ["GENERATOR_MODEL"]
JUDGE_MODEL        = os.environ["JUDGE_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generator model : {GENERATOR_MODEL}")
print(f"Judge model     : {JUDGE_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

Provider        : ollama
Embedding model : nomic-embed-text-v2-moe:latest
Generator model : mistral-small3.2:24b
Judge model     : llama3.1:8b-instruct-q4_K_M
Chroma dir      : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [5]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)

llm = create_llm(PROVIDER, "mistral-small3.2:24b")

judge_llm = create_llm(PROVIDER, JUDGE_MODEL, temperature=0)

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")

### 2. Load CUAD Samples from RAGBench

N=100 this time — large enough to average out the per-sample noise that made the N=20 numbers
hard to interpret.

In [6]:
N_SAMPLES = 100

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))


def fmt(v):
    return f"{v:.3f}" if v is not None else "N/A"


print(f"Loaded {len(samples)} CUAD samples")

s = samples[0]
print(f"\nQuestion         : {s['question']}")
print(f"Doc length       : {len(s['documents'][0])} chars")
print(f"Adherence score  : {s['adherence_score']}")
print(f"Relevance score  : {fmt(s['relevance_score'])}")
print(f"Utilization score: {fmt(s['utilization_score'])}")
print(f"Completeness     : {fmt(s['completeness_score'])}")

Loaded 100 CUAD samples

Question         : Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Doc length       : 122054 chars
Adherence score  : True
Relevance score  : 0.000
Utilization score: 0.000
Completeness     : 1.000


### 3. Run Experiment N config (HyDE + nomic + mistral-small3.2:24b + PROMPT_V2) at N=100

In [7]:
# Experiment N @ N=100 — HyDE + nomic + PROMPT_V2 + mistral-small3.2:24b.
# Identical config to notebook 02's Experiment N; only N_SAMPLES changed (20 -> 100).
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_n100"

results_n100 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=llm,
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)


[1/100] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.337s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 5.970s
[timing] llm      : 1.276s
[timing] llm      : 27.492s
  our: Absent.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/100] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 2.603s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 5.389s
[timing] llm      : 1.448s
[timing] llm      : 24.522s
  our: Absent.
  ref: No, the contract does not contain a license granted by one party to its counterparty. The contract is a Non-Competition Agreement and Right of First Offer between Glamis Gold Ltd. and Western Copper Corporation. It does not involve the granting of any lice

In [9]:
# Experiment N @ N=100 — TRACe evaluation. n_runs=1: dataset size is the variance-reduction
# mechanism at this scale (see Retrieval_Strategy.md "Notes for Scaling Up").
JUDGE_N_RUNS = 1
agg_n100 = evaluate_results(results_n100, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Exp N @ N=100 (HyDE + PROMPT_V2 + mistral-small3.2) — Rel {agg_n100['avg_relevance']:.3f} / Util {agg_n100['avg_utilization']:.3f} / Comp {agg_n100['avg_completeness']:.3f} / Adh {agg_n100['adherence_rate']:.0%}")

[1/100] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The question asks about deposit
  Relevance   : 0.334 — The relevant information for answering this question can be found in Section 13 of the doc
  Utilization : 0.046
  Completeness: 0.138

[_strip_trailing_commas] trailing comma stripped from judge output
[2/100] [FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The response claims that there 
  Relevance   : 0.183 — The relevant information for answering this question can be found in PART 1 of the contrac
  Utilization : 0.000
  Completeness: 0.000

[3/100] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The response claims that the co
  Relevance   : 0.051 — The doc

### 4. Step N diagnosis — isolate run-to-run noise from a real `top_k` effect

Per-sample analysis of the run above showed two separate things:

1. **Non-determinism dominates the N=20 → N=100 swing.** `hyde_llm` and the generator run at
   Ollama's default (non-zero) temperature — only `judge_llm` is pinned to `temperature=0`.
   Re-scoring just the first 20 samples of this same N=100 run gave avg relevance **0.188**,
   not the **0.540** reported for Experiment N on the same 20 questions — same code, same
   config, different random draws for the HyDE hypothesis and the generated answer.
2. **A structural ceiling, independent of variance.** 27/100 samples show
   `relevance ≈ utilization` with `completeness ≈ 1.0` — the judge finds a small relevant span
   fully captured, but relevance stays low because the denominator is the full fixed
   `top_k=20` context regardless of how small that span is. Doc length is not the cause
   (correlation between chunk count and relevance/utilization ≈ 0 across all 100 samples).

Four runs below, in order, all sharing the same deterministic (temp=0) baseline so each isolates
exactly one variable:

- **4a. temp=0 re-baseline** — identical config to Section 3 (`top_k=20`). Isolates how much of
  the swing above was pure sampling noise.
- **4b. temp=0 + top_k=10** — `top_k` reduced 20 → 10. First point on the top_k sweep.
- **4c. temp=0 + top_k=5** — third point on the top_k sweep (20 → 10 → 5), grouped with 4a/4b.
- **4d. temp=0 + sentence-level chunking** — `top_k=20`, `SentenceTextSplitter` instead of
  `RecursiveCharacterTextSplitter`. Isolates the chunking effect from the top_k effect.

In [10]:
# Deterministic HyDE + generator — only the judge was pinned to temperature=0 before this.
llm_det = create_llm(PROVIDER, "mistral-small3.2:24b", temperature=0)
hyde_llm_det = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M", temperature=0)

In [11]:
# 4a. temp=0 re-baseline — same top_k=20 as Section 3, deterministic HyDE + generator.
TOP_K_BASELINE = 20
COLLECTION_TAG_TEMP0 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_n100_temp0_k{TOP_K_BASELINE}"

results_n100_temp0 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm_det, top_k=TOP_K_BASELINE),
    embeddings=embeddings,
    generator_llm=llm_det,
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_TEMP0,
    retrieve_label=f"hyde retrieve (top-{TOP_K_BASELINE}, temp=0)",
)


[1/100] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.378s  (291 chunks embedded)
[timing] hyde retrieve (top-20, temp=0) : 3.818s
[timing] llm      : 2.376s
[timing] llm      : 27.185s
  our: Absent.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/100] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 2.766s  (47 chunks embedded)
[timing] hyde retrieve (top-20, temp=0) : 6.007s
[timing] llm      : 1.829s
[timing] llm      : 26.568s
  our: The context provided does not contain any clause granting a license from one party to its counterparty. Therefore, it is absent.
  ref: No, the contract does not contain a license granted by one party to its counterparty. The contract is a Non-Competition

In [12]:
# 4a. TRACe evaluation for the temp=0 re-baseline.
agg_n100_temp0 = evaluate_results(results_n100_temp0, judge_llm, n_runs=1)
print(f"Exp N @ N=100, temp=0, top_k={TOP_K_BASELINE} — Rel {agg_n100_temp0['avg_relevance']:.3f} / "
      f"Util {agg_n100_temp0['avg_utilization']:.3f} / Comp {agg_n100_temp0['avg_completeness']:.3f} / "
      f"Adh {agg_n100_temp0['adherence_rate']:.0%}")

[1/100] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The response claims that one pa
  Relevance   : 0.133 — The relevant information for answering this question can be found in document 3, which dis
  Utilization : 0.000
  Completeness: 0.000

[2/100] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response claims that there is no clause granting a license from one party to its count
  Relevance   : 0.061 — The relevant information for answering this question can be found in document 1, specifica
  Utilization : 0.061
  Completeness: 1.000

[3/100] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported because it correctly states that there is no specific
  Relevance   : 0.034 — The relevant document for answering this question is 0a, which mentions th

In [13]:
# 4b. temp=0 + top_k=10 — same deterministic setup, smaller fixed context window.
TOP_K_REDUCED = 10
COLLECTION_TAG_TEMP0_K10 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_n100_temp0_k{TOP_K_REDUCED}"

results_n100_temp0_k10 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm_det, top_k=TOP_K_REDUCED),
    embeddings=embeddings,
    generator_llm=llm_det,
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_TEMP0_K10,
    retrieve_label=f"hyde retrieve (top-{TOP_K_REDUCED}, temp=0)",
)


[1/100] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 6.222s  (291 chunks embedded)
[timing] hyde retrieve (top-10, temp=0) : 3.306s
[timing] llm      : 3.765s
[timing] llm      : 23.774s
  our: Absent.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/100] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 2.512s  (47 chunks embedded)
[timing] hyde retrieve (top-10, temp=0) : 6.375s
[timing] llm      : 1.823s
[timing] llm      : 20.335s
  our: The context provided does not contain any clause granting a license from one party to its counterparty. Therefore, it is absent.
  ref: No, the contract does not contain a license granted by one party to its counterparty. The contract is a Non-Competition

In [14]:
# 4b. TRACe evaluation for temp=0 + top_k=10.
agg_n100_temp0_k10 = evaluate_results(results_n100_temp0_k10, judge_llm, n_runs=1)
print(f"Exp N @ N=100, temp=0, top_k={TOP_K_REDUCED} — Rel {agg_n100_temp0_k10['avg_relevance']:.3f} / "
      f"Util {agg_n100_temp0_k10['avg_utilization']:.3f} / Comp {agg_n100_temp0_k10['avg_completeness']:.3f} / "
      f"Adh {agg_n100_temp0_k10['adherence_rate']:.0%}")

[_strip_trailing_commas] trailing comma stripped from judge output
[1/100] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The question asks about deposit
  Relevance   : 0.375 — The relevant information for answering this question can be found in document 3, which dis
  Utilization : 0.065
  Completeness: 0.174

[2/100] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response claims that the contract does not contain a license granted by one party to i
  Relevance   : 0.223 — The relevant information for answering this question can be found in document 1, specifica
  Utilization : 0.223
  Completeness: 1.000

[3/100] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported because it correctly identifies that the context does
  Relevance   : 0.075 — The rel

### 4c. top_k sweep — third point at top_k=5

4a (`top_k=20`) and 4b (`top_k=10`) give two points on the top_k curve. A third point at
`top_k=5` checks whether relevance/utilization keep improving as `top_k` shrinks or whether it
plateaus/reverses (too little context can cost completeness). Same deterministic setup as 4b.

In [15]:
# 4c. temp=0 + top_k=5 — third point on the top_k sweep.
TOP_K_MIN = 5
COLLECTION_TAG_TEMP0_K5 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_n100_temp0_k{TOP_K_MIN}"

results_n100_temp0_k5 = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm_det, top_k=TOP_K_MIN),
    embeddings=embeddings,
    generator_llm=llm_det,
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_TEMP0_K5,
    retrieve_label=f"hyde retrieve (top-{TOP_K_MIN}, temp=0)",
)


[1/100] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 6.345s  (291 chunks embedded)
[timing] hyde retrieve (top-5, temp=0) : 3.203s
[timing] llm      : 3.787s
[timing] llm      : 18.451s
  our: Absent.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/100] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 2.589s  (47 chunks embedded)
[timing] hyde retrieve (top-5, temp=0) : 6.468s
[timing] llm      : 1.814s
[timing] llm      : 15.479s
  our: The context provided does not contain information about a license granted by one party to its counterparty. Therefore, it is absent.
  ref: No, the contract does not contain a license granted by one party to its counterparty. The contract is a Non-Competiti

In [16]:
# 4c. TRACe evaluation for temp=0 + top_k=5.
agg_n100_temp0_k5 = evaluate_results(results_n100_temp0_k5, judge_llm, n_runs=1)
print(f"Exp N @ N=100, temp=0, top_k={TOP_K_MIN} — Rel {agg_n100_temp0_k5['avg_relevance']:.3f} / "
      f"Util {agg_n100_temp0_k5['avg_utilization']:.3f} / Comp {agg_n100_temp0_k5['avg_completeness']:.3f} / "
      f"Adh {agg_n100_temp0_k5['adherence_rate']:.0%}")

[_strip_trailing_commas] trailing comma stripped from judge output
[1/100] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The response claims that there 
  Relevance   : 0.300 — The relevant information for answering this question can be found in document 1, specifica
  Utilization : 0.300
  Completeness: 1.000

[2/100] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.673 — The relevant document for answering this question is 3a, which contains information about 
  Utilization : 0.673
  Completeness: 1.000

[3/100] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — Claim a: The context does not specify a particular date when the contract is effective. Th
  Relevance   : 0.125 — The rel

### 4d. Sentence-level chunking (deterministic, top_k=20)

`RecursiveCharacterTextSplitter` splits on character count and can cut a CUAD clause mid-sentence
(`Retrieval_Strategy.md`, "Next Steps" #3). `SentenceTextSplitter`
(`src/reliablerag/chunking.py`) merges whole sentences up to `chunk_size` instead, so chunk
boundaries always fall between sentences. Run on the same deterministic (temp=0) setup as 4a/4b,
`top_k=20`, to isolate the chunking effect from the top_k effect.

In [19]:
# 4d. Sentence-level chunking — same chunk_size, deterministic HyDE + generator, top_k=20.
sentence_splitter = SentenceTextSplitter(chunk_size=CHUNK_SIZE)
COLLECTION_TAG_SENTCHUNK = f"sentchunk_cs{CHUNK_SIZE}_n100_temp0_k{TOP_K_BASELINE}"

results_n100_sentchunk = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm_det, top_k=TOP_K_BASELINE),
    embeddings=embeddings,
    generator_llm=llm_det,
    prompt_template=PROMPT_V2,
    splitter=sentence_splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_SENTCHUNK,
    retrieve_label=f"hyde retrieve (top-{TOP_K_BASELINE}, sentence chunks)",
)


[1/100] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.308s  (234 chunks embedded)
[timing] hyde retrieve (top-20, sentence chunks) : 5.606s
[timing] llm      : 2.362s
[timing] llm      : 37.939s
  our: Absent.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/100] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 3.488s  (30 chunks embedded)
[timing] hyde retrieve (top-20, sentence chunks) : 5.773s
[timing] llm      : 1.810s
[timing] llm      : 76.321s
  our: Absent.
  ref: No, the contract does not contain a license granted by one party to its counterparty. The contract is a Non-Competition Agreement and Right of First Offer between Glamis Gold Ltd. and Western Copper Corporation. It does no

In [20]:
# 4d. TRACe evaluation for sentence-level chunking.
agg_n100_sentchunk = evaluate_results(results_n100_sentchunk, judge_llm, n_runs=1)
print(f"Exp N @ N=100, temp=0, top_k={TOP_K_BASELINE}, sentence chunks — "
      f"Rel {agg_n100_sentchunk['avg_relevance']:.3f} / "
      f"Util {agg_n100_sentchunk['avg_utilization']:.3f} / Comp {agg_n100_sentchunk['avg_completeness']:.3f} / "
      f"Adh {agg_n100_sentchunk['adherence_rate']:.0%}")

[1/100] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The question asks about deposit
  Relevance   : 0.722 — The relevant information for answering this question can be found in Articles III and XIII
  Utilization : 0.000
  Completeness: 0.000

[_strip_trailing_commas] trailing comma stripped from judge output
[2/100] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The question asks if the contra
  Relevance   : 0.307 — The relevant information for answering this question can be found in Part 1 of the contrac
  Utilization : 0.188
  Completeness: 0.613

[_strip_trailing_commas] trailing comma stripped from judge output
[3/100] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response is supported by the documents. The claim t

In [21]:
# Summary — original (non-deterministic) run vs all Step N diagnostic runs.
rows = [
    ("N=100, non-det, top_k=20 (Section 3)",   agg_n100),
    ("N=100, temp=0, top_k=20",                agg_n100_temp0),
    ("N=100, temp=0, top_k=10",                agg_n100_temp0_k10),
    ("N=100, temp=0, top_k=5",                 agg_n100_temp0_k5),
    ("N=100, temp=0, top_k=20, sent. chunks",  agg_n100_sentchunk),
]
print(f"{'Config':<42}{'Rel':>8}{'Util':>8}{'Comp':>8}{'Adh':>8}")
for label, agg in rows:
    print(f"{label:<42}{agg['avg_relevance']:>8.3f}{agg['avg_utilization']:>8.3f}"
          f"{agg['avg_completeness']:>8.3f}{agg['adherence_rate']:>8.0%}")

Config                                         Rel    Util    Comp     Adh
N=100, non-det, top_k=20 (Section 3)         0.165   0.073   0.535     51%
N=100, temp=0, top_k=20                      0.174   0.070   0.432     52%
N=100, temp=0, top_k=10                      0.287   0.112   0.403     48%
N=100, temp=0, top_k=5                       0.422   0.183   0.469     47%
N=100, temp=0, top_k=20, sent. chunks        0.236   0.094   0.465     59%
